# Decision Support Evidence With PySpark SQL

This Kaggle notebook turns the final fake-review experiment into reproducible decision-support evidence using PySpark DataFrames and `spark.sql()` temporary views. It supports review prioritization, error analysis, data collection, duplicate audits, drift monitoring, and pseudo-labeling assessment. It does not prove that a seller is fraudulent or that a feature causes deception. High-impact actions require human review and additional evidence.

Label convention used by the final run: `0 = fake`, `1 = genuine`.


In [1]:
import os
from pathlib import Path
from functools import reduce
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.functions import vector_to_array

spark = (SparkSession.builder.appName('bangla-fake-review-decision-support')
    .config('spark.sql.shuffle.partitions', '200').getOrCreate())
spark.sparkContext.setLogLevel('WARN')

def candidate_roots():
    configured = os.environ.get('PROJECT_ROOT')
    bases = [Path(configured)] if configured else []
    bases += [
        Path('/kaggle/input/notebooks/borshon47/evalu'),
        Path('/kaggle/input/notebooks/borshon47/featureengi'),
        Path('/kaggle/input/notebooks/borshon47/preprocess1'),
        Path('/kaggle/input'),
        Path('/kaggle/working'),
        Path.cwd(),
    ]
    seen = set()
    for base in bases:
        if not base or not base.exists():
            continue
        for path in [base, base / 'bangla_fake_review', base / 'Final Output' / 'bangla_fake_review']:
            if path.exists() and path.name == 'bangla_fake_review':
                key = str(path.resolve())
                if key not in seen:
                    seen.add(key)
                    yield path
        if base.is_dir():
            for path in base.rglob('bangla_fake_review'):
                if path.is_dir():
                    key = str(path.resolve())
                    if key not in seen:
                        seen.add(key)
                        yield path

def find_artifact_dir(required_child, preferred_names):
    matches = []
    for root in candidate_roots():
        if (root / required_child).exists():
            score = 0
            root_text = str(root).lower()
            for weight, name in enumerate(preferred_names[::-1], start=1):
                if name.lower() in root_text:
                    score += weight
            matches.append((score, root))
    if not matches:
        searched = [str(p) for p in candidate_roots()]
        raise FileNotFoundError(f'Could not find a bangla_fake_review folder containing {required_child}. Searched: {searched}')
    return sorted(matches, key=lambda item: item[0], reverse=True)[0][1]

ARTIFACT_ROOTS = list(candidate_roots())
PROCESSED_ARTIFACT_ROOT = find_artifact_dir('processed', ['featureengi', 'preprocess1', 'evalu'])
RESULT_ARTIFACT_ROOT = find_artifact_dir('results', ['evalu', 'featureengi', 'preprocess1'])
OUT_ROOT = Path('/kaggle/working/decision_support_outputs')
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('PROCESSED_ARTIFACT_ROOT =', PROCESSED_ARTIFACT_ROOT)
print('RESULT_ARTIFACT_ROOT =', RESULT_ARTIFACT_ROOT)
print('ALL_ARTIFACT_ROOTS =')
for root in ARTIFACT_ROOTS:
    print(' -', root)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/21 10:41:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PROCESSED_ARTIFACT_ROOT = /kaggle/input/notebooks/borshon47/featureengi/bangla_fake_review
RESULT_ARTIFACT_ROOT = /kaggle/input/notebooks/borshon47/evalu/bangla_fake_review
ALL_ARTIFACT_ROOTS =
 - /kaggle/input/notebooks/borshon47/evalu/bangla_fake_review
 - /kaggle/input/notebooks/borshon47/featureengi/bangla_fake_review
 - /kaggle/input/notebooks/borshon47/preprocess1/bangla_fake_review


## Load final-run artifacts

The notebook reads final machine-readable parquet and CSV artifacts and fails loudly if the Kaggle upload is incomplete.


In [2]:
def find_artifact(name, kind):
    preferred = {
        'processed': ['featureengi', 'preprocess1', 'evalu'],
        'results': ['evalu', 'featureengi', 'preprocess1'],
    }[kind]
    matches = []
    for root in ARTIFACT_ROOTS:
        direct = root / kind / name
        if direct.exists():
            matches.append(direct)
        for path in root.rglob(name):
            if path.exists() and path not in matches:
                matches.append(path)
    if not matches:
        searched = [str(root) for root in ARTIFACT_ROOTS]
        raise FileNotFoundError(f'Could not find {name}. Searched artifact roots: {searched}')
    def score(path):
        text = str(path).lower()
        return sum((i + 1) for i, item in enumerate(preferred[::-1]) if item.lower() in text)
    best = sorted(matches, key=score, reverse=True)[0]
    print(f'{name} -> {best}')
    return str(best)

test_features = spark.read.parquet(find_artifact('test_features.parquet', 'processed'))
train_features = spark.read.parquet(find_artifact('train_features.parquet', 'processed'))
validation_features = spark.read.parquet(find_artifact('validation_features.parquet', 'processed'))
test_gold = spark.read.parquet(find_artifact('test_predictions_gold.parquet', 'results'))
test_expanded = spark.read.parquet(find_artifact('test_predictions_expanded.parquet', 'results'))
metrics = spark.read.option('header', True).option('inferSchema', True).csv(find_artifact('test_results_final.csv', 'results'))
try:
    gold_validation = spark.read.option('header', True).option('inferSchema', True).csv(find_artifact('gold_validation_metrics.csv', 'results'))
except FileNotFoundError:
    print('gold_validation_metrics.csv not found; skipping validation-model comparison.')
    gold_validation = spark.createDataFrame([], 'model string, macro_f1 double, fake_pr_auc double, fake_f1 double, accuracy double')
for name, df in [('test_features', test_features), ('train_features', train_features), ('validation_features', validation_features), ('test_metrics', metrics), ('gold_validation_metrics', gold_validation)]:
    df.createOrReplaceTempView(name); print(name, df.count())


test_features.parquet -> /kaggle/input/notebooks/borshon47/preprocess1/bangla_fake_review/processed/test_features.parquet


train_features.parquet -> /kaggle/input/notebooks/borshon47/preprocess1/bangla_fake_review/processed/train_features.parquet
validation_features.parquet -> /kaggle/input/notebooks/borshon47/preprocess1/bangla_fake_review/processed/validation_features.parquet
test_predictions_gold.parquet -> /kaggle/input/notebooks/borshon47/evalu/bangla_fake_review/results/test_predictions_gold.parquet
test_predictions_expanded.parquet -> /kaggle/input/notebooks/borshon47/evalu/bangla_fake_review/results/test_predictions_expanded.parquet
test_results_final.csv -> /kaggle/input/notebooks/borshon47/evalu/bangla_fake_review/results/test_results_final.csv
gold_validation_metrics.csv not found; skipping validation-model comparison.


26/09/21 10:41:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


test_features 1356
train_features 6321
validation_features 1356
test_metrics 2


gold_validation_metrics 0


## 1. Data quality and class balance

Supports the use of macro metrics, class weighting, and a locked test split.


In [3]:
class_balance = spark.sql('''
SELECT 'train' AS split, label, COUNT(*) AS rows,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY 'train'), 2) AS pct
FROM train_features GROUP BY label
UNION ALL
SELECT 'validation', label, COUNT(*), ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY 'validation'), 2)
FROM validation_features GROUP BY label
UNION ALL
SELECT 'test', label, COUNT(*), ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY 'test'), 2)
FROM test_features GROUP BY label ORDER BY split, label
''')
class_balance.show(); class_balance.coalesce(1).write.mode('overwrite').option('header', True).csv(str(OUT_ROOT / 'class_balance'))


26/09/21 10:41:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/21 10:41:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/21 10:41:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/21 10:41:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/21 10:41:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/21 10:41:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/21 1

+----------+-----+----+-----+
|     split|label|rows|  pct|
+----------+-----+----+-----+
|      test|    0| 201|14.82|
|      test|    1|1155|85.18|
|     train|    0| 934|14.78|
|     train|    1|5387|85.22|
|validation|    0| 201|14.82|
|validation|    1|1155|85.18|
+----------+-----+----+-----+



26/09/21 10:41:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/21 10:41:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/21 10:41:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/21 10:41:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/21 10:41:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/21 10:41:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/21 1

## 2. Model selection and pseudo-labeling decisions

These queries reproduce the claims that Random Forest was selected and that pseudo-label expansion reduced locked-test performance.


In [4]:
if gold_validation.count() > 0:
    spark.sql('''
SELECT model, ROUND(macro_f1, 4) AS macro_f1, ROUND(fake_pr_auc, 4) AS fake_pr_auc,
       ROUND(fake_f1, 4) AS fake_f1, ROUND(accuracy, 4) AS accuracy
FROM gold_validation_metrics ORDER BY macro_f1 DESC, fake_pr_auc DESC
    ''').show(truncate=False)
else:
    print('No gold validation metrics available; continuing with locked-test comparison.')

test_decision = spark.sql('''
SELECT condition, model, ROUND(macro_f1, 4) AS macro_f1, ROUND(accuracy, 4) AS accuracy,
       ROUND(fake_pr_auc, 4) AS fake_pr_auc, ROUND(fake_recall, 4) AS fake_recall,
       ROUND(fake_f1, 4) AS fake_f1 FROM test_metrics ORDER BY macro_f1 DESC
''')
test_decision.show(truncate=False); test_decision.coalesce(1).write.mode('overwrite').option('header', True).csv(str(OUT_ROOT / 'test_decision_comparison'))


No gold validation metrics available; continuing with locked-test comparison.
+----------------+-------------+--------+--------+-----------+-----------+-------+
|condition       |model        |macro_f1|accuracy|fake_pr_auc|fake_recall|fake_f1|
+----------------+-------------+--------+--------+-----------+-----------+-------+
|gold_only       |random_forest|0.9194  |0.9617  |0.9213     |0.801      |0.861  |
|gold_plus_pseudo|random_forest|0.9098  |0.958   |0.9033     |0.7662     |0.8438 |
+----------------+-------------+--------+--------+-----------+-----------+-------+



## 3. Human-review risk queue

Ranks reviews for analyst investigation. It is not an automatic enforcement decision.


In [5]:
def with_fake_probability(df):
    if 'probability' in df.columns: return df.withColumn('fake_probability', vector_to_array('probability')[0])
    if 'rawPrediction' in df.columns:
        raw = vector_to_array('rawPrediction')
        return (df.withColumn('_raw_values', raw)
                  .withColumn('_raw_total', F.col('_raw_values')[0] + F.col('_raw_values')[1])
                  .withColumn('fake_probability',
                              F.when(F.col('_raw_total') > 0,
                                     F.col('_raw_values')[0] / F.col('_raw_total'))
                               .otherwise(F.lit(0.0)))
                  .drop('_raw_values', '_raw_total'))
    if 'probability_fake' in df.columns: return df.withColumnRenamed('probability_fake', 'fake_probability')
    if 'fake_probability' in df.columns: return df
    raise ValueError('No probability column found in prediction artifact')

risk = with_fake_probability(test_expanded); risk.createOrReplaceTempView('risk_predictions')
risk_queue = spark.sql('''
SELECT review_id, label, prediction, ROUND(fake_probability, 4) AS fake_probability,
       CASE WHEN fake_probability >= 0.90 THEN 'high_priority'
            WHEN fake_probability >= 0.75 THEN 'review_sample' ELSE 'low_priority' END AS review_queue
FROM risk_predictions ORDER BY fake_probability DESC
''')
risk_queue.show(20, truncate=False); risk_queue.coalesce(1).write.mode('overwrite').option('header', True).csv(str(OUT_ROOT / 'human_review_queue'))


+----------------------------------------------------------------+-----+----------+----------------+-------------+
|review_id                                                       |label|prediction|fake_probability|review_queue |
+----------------------------------------------------------------+-----+----------+----------------+-------------+
|9ccbf03b6d1b06d9aeaea49d108c41cdf125a43e3c61dabb431bc478e66ab4bc|0    |0.0       |0.998           |high_priority|
|61bcd85eff20d00587737757b55c75fe4cece41cdbc02dd34a0141edc4769b03|0    |0.0       |0.9973          |high_priority|
|889034036c40961d53fa2872df4bf9ef8b110cefe1f35e73e6ff670657488bb0|0    |0.0       |0.9972          |high_priority|
|b72f4451c206a46488d4b242f5b71c243c6315d93903247613c8c1cabe22b329|0    |0.0       |0.9971          |high_priority|
|3ed24f4cbc033a25879288ab1a4735da96c1c1d342829f3679ea3e41b3d20ba2|0    |0.0       |0.997           |high_priority|
|ff558eb178ccf1f654bedb076dad315f816938a149b85a44a8e6342f6119811f|0    |0.0     

## 4. Error analysis for data collection

This identifies segments for additional annotation. Differences are descriptive patterns, not causal rules.


In [6]:
analysis = test_expanded
for c in ['char_length', 'token_count', 'codemix_ratio', 'promo_keyword_hits', 'near_duplicate_degree', 'has_promo']:
    if c in test_features.columns and c not in analysis.columns:
        analysis = analysis.join(test_features.select('review_id', c), 'review_id', 'left')
for c in ['char_length', 'token_count', 'codemix_ratio', 'promo_keyword_hits', 'near_duplicate_degree']:
    if c not in analysis.columns:
        analysis = analysis.withColumn(c, F.lit(None).cast('double'))
analysis = analysis.withColumn('error_type', F.when(F.col('label') == F.col('prediction'), 'correct').when((F.col('label') == 0) & (F.col('prediction') == 1), 'false_negative').otherwise('false_positive'))
analysis.createOrReplaceTempView('error_analysis')
error_profile = spark.sql('''
SELECT error_type, COUNT(*) AS rows, ROUND(AVG(char_length), 2) AS mean_chars,
       ROUND(AVG(token_count), 2) AS mean_tokens, ROUND(AVG(codemix_ratio), 4) AS mean_codemix,
       ROUND(AVG(promo_keyword_hits), 4) AS mean_promo_hits,
       ROUND(AVG(near_duplicate_degree), 4) AS mean_near_duplicate_degree
FROM error_analysis GROUP BY error_type ORDER BY error_type
''')
error_profile.show(truncate=False); error_profile.coalesce(1).write.mode('overwrite').option('header', True).csv(str(OUT_ROOT / 'error_profile'))


+--------------+----+----------+-----------+------------+---------------+--------------------------+
|error_type    |rows|mean_chars|mean_tokens|mean_codemix|mean_promo_hits|mean_near_duplicate_degree|
+--------------+----+----------+-----------+------------+---------------+--------------------------+
|correct       |1299|674.71    |101.51     |0.1442      |0.3788         |NULL                      |
|false_negative|47  |450.43    |69.02      |0.1327      |0.7234         |NULL                      |
|false_positive|10  |746.3     |118.6      |0.1773      |1.0            |NULL                      |
+--------------+----+----------+-----------+------------+---------------+--------------------------+



## 5. Duplicate exposure and leakage-control evidence

Duplicate exposure supports duplicate-aware splitting and audit fields. It is not proof of coordinated behavior.


In [7]:
dup_cols = [c for c in ['review_id', 'exact_dup_cluster_size', 'is_exact_duplicate', 'near_duplicate_degree', 'near_duplicate_similarity', 'near_duplicate_cluster_size'] if c in test_features.columns]
if len(dup_cols) > 1:
    duplicate_exprs = [F.col('review_id')]
    for c in ['exact_dup_cluster_size', 'is_exact_duplicate', 'near_duplicate_degree', 'near_duplicate_similarity', 'near_duplicate_cluster_size']:
        if c in test_features.columns:
            duplicate_exprs.append(F.col(c))
        else:
            duplicate_exprs.append(F.lit(None).cast('double').alias(c))
    test_features.select(*duplicate_exprs).createOrReplaceTempView('test_duplicate_exposure')
    duplicate_summary = spark.sql('''
    SELECT COUNT(*) AS rows, SUM(CASE WHEN is_exact_duplicate = 1 THEN 1 ELSE 0 END) AS exact_duplicate_rows,
           ROUND(AVG(exact_dup_cluster_size), 4) AS mean_exact_cluster_size,
           ROUND(AVG(near_duplicate_degree), 4) AS mean_near_duplicate_degree,
           ROUND(MAX(near_duplicate_degree), 4) AS max_near_duplicate_degree
    FROM test_duplicate_exposure
    ''')
    duplicate_summary.show(truncate=False); duplicate_summary.coalesce(1).write.mode('overwrite').option('header', True).csv(str(OUT_ROOT / 'duplicate_exposure'))
else: print('Duplicate feature columns were not found in test_features')


+----+--------------------+-----------------------+--------------------------+-------------------------+
|rows|exact_duplicate_rows|mean_exact_cluster_size|mean_near_duplicate_degree|max_near_duplicate_degree|
+----+--------------------+-----------------------+--------------------------+-------------------------+
|1356|4                   |1.0029                 |NULL                      |NULL                     |
+----+--------------------+-----------------------+--------------------------+-------------------------+



## 6. Segment monitoring and drift indicators

These statistics indicate possible distribution change. They should trigger investigation, not automatic retraining.


In [8]:
monitor_cols = [c for c in ['char_length', 'token_count', 'codemix_ratio', 'promo_keyword_hits', 'emoji_count', 'has_promo'] if c in train_features.columns and c in validation_features.columns and c in test_features.columns]
rows = []
for c in monitor_cols:
    rows.append(spark.sql(f'''SELECT '{c}' AS feature, (SELECT AVG(CAST({c} AS DOUBLE)) FROM train_features) AS train_mean, (SELECT AVG(CAST({c} AS DOUBLE)) FROM validation_features) AS validation_mean, (SELECT AVG(CAST({c} AS DOUBLE)) FROM test_features) AS test_mean'''))
if rows:
    drift = reduce(lambda a, b: a.unionByName(b), rows).select('feature', F.round('train_mean', 4).alias('train_mean'), F.round('validation_mean', 4).alias('validation_mean'), F.round('test_mean', 4).alias('test_mean'))
    drift.show(truncate=False); drift.coalesce(1).write.mode('overwrite').option('header', True).csv(str(OUT_ROOT / 'drift_indicators'))
else: print('No common monitoring columns were found')


+------------------+----------+---------------+---------+
|feature           |train_mean|validation_mean|test_mean|
+------------------+----------+---------------+---------+
|char_length       |659.1758  |654.9963       |667.4668 |
|token_count       |99.037    |98.4292        |100.514  |
|codemix_ratio     |0.1432    |0.1469         |0.1441   |
|promo_keyword_hits|0.427     |0.4395         |0.3953   |
|emoji_count       |1.171     |1.2721         |1.0929   |
|has_promo         |0.3205    |0.3311         |0.2994   |
+------------------+----------+---------------+---------+



## Interpretation rules

- A risk queue is for analyst prioritization, not automatic punishment.
- Error profiles identify where more labels are valuable; they do not establish causality.
- Drift indicates possible distribution change; it does not prove model failure.
- Duplicate summaries justify leakage controls; they do not prove coordination.
- Final enforcement decisions require human review, policy evidence, and an appeal path.

Outputs are written under `/kaggle/working/decision_support_outputs`.
